# API Ingestion Demo — DummyJSON Products API

```text
API endpoint
↓
Call API
↓
Save raw JSON
↓
Convert JSON to DataFrame
↓
Flatten nested JSON
↓
Validate required fields
↓
Separate required-field missing values and optional-field missing values
↓
Remove duplicate rows
↓
Save staging output
↓
Save clean output
↓
Generate ingestion log
```

## API Source

```text
https://dummyjson.com/products
```

## Expected Outputs

```text
data/raw/api/dummyjson_products_raw.json
data/staging/api/dummyjson_products_staging.csv
data/clean/api/dummyjson_products_clean.csv
logs/api_ingestion_log.json
```

## 1. Import Required Libraries

In [6]:
import pandas as pd
import json
import uuid
import re
import requests

from pathlib import Path
from datetime import datetime, timezone

## 2. Define Project Paths

In [7]:
def find_project_root(current_path: Path) -> Path:
    current_path = current_path.resolve()

    for path in [current_path] + list(current_path.parents):
        if (path / "data").exists():
            return path

    raise FileNotFoundError("Project root not found. Make sure data/ folder exists.")


def to_relative_path(path: Path, project_root: Path) -> str:
    return path.resolve().relative_to(project_root.resolve()).as_posix()


CURRENT_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(CURRENT_DIR)

api_url = "https://dummyjson.com/products"

raw_dir = PROJECT_ROOT / "data" / "raw" / "api"
staging_dir = PROJECT_ROOT / "data" / "staging" / "api"
clean_dir = PROJECT_ROOT / "data" / "clean" / "api"
log_dir = PROJECT_ROOT / "logs"

raw_dir.mkdir(parents=True, exist_ok=True)
staging_dir.mkdir(parents=True, exist_ok=True)
clean_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

raw_output_path = raw_dir / "dummyjson_products_raw.json"
staging_output_path = staging_dir / "dummyjson_products_staging.csv"
clean_output_path = clean_dir / "dummyjson_products_clean.csv"
log_output_path = log_dir / "api_ingestion_log.json"

print("Current dir:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("API URL:", api_url)
print("Raw output:", raw_output_path)
print("Staging output:", staging_output_path)
print("Clean output:", clean_output_path)
print("Log output:", log_output_path)

Current dir: f:\data\new\quanskill\DataVision_Duy\week2\notebooks\data_team
Project root: F:\data\new\quanskill\DataVision_Duy\week2
API URL: https://dummyjson.com/products
Raw output: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\api\dummyjson_products_raw.json
Staging output: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\api\dummyjson_products_staging.csv
Clean output: F:\data\new\quanskill\DataVision_Duy\week2\data\clean\api\dummyjson_products_clean.csv
Log output: F:\data\new\quanskill\DataVision_Duy\week2\logs\api_ingestion_log.json


## 3. Helper Functions

In [8]:
def clean_column_name(column_name: str) -> str:
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    return column_name.strip("_")


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_column_name(col) for col in df.columns]
    return df


def flatten_list_value(value):
    if isinstance(value, list):
        return json.dumps(value, ensure_ascii=False)
    return value


def flatten_api_products(api_response: dict) -> pd.DataFrame:
    products = api_response.get("products", [])

    df = pd.json_normalize(
        products,
        sep="_"
    )

    for col in df.columns:
        df[col] = df[col].apply(flatten_list_value)

    df = clean_columns(df)

    return df

## 4. Start Ingestion Run

In [9]:
run_id = str(uuid.uuid4())
source_name = "dummyjson_products_api"
source_type = "api"
owner = "Nguyen Minh Duy"

start_time = datetime.now(timezone.utc).isoformat()

print("Run ID:", run_id)
print("Start time:", start_time)

Run ID: 0ef8141c-6f3c-4a3c-b8ed-4931d2dcba6f
Start time: 2026-07-01T06:32:21.546195+00:00


## 5. Call API

In [10]:
try:
    response = requests.get(
        api_url,
        timeout=30
    )

    response.raise_for_status()

    api_response = response.json()

    status = "success"
    error_message = None

    print("API request successful.")
    print("Status code:", response.status_code)
    print("Top-level keys:", list(api_response.keys()))

except Exception as error:
    status = "failed"
    error_message = str(error)
    raise

API request successful.
Status code: 200
Top-level keys: ['products', 'total', 'skip', 'limit']


## 6. Save Raw JSON

In [11]:
with open(raw_output_path, "w", encoding="utf-8") as file:
    json.dump(
        api_response,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Raw JSON saved:", to_relative_path(raw_output_path, PROJECT_ROOT))

Raw JSON saved: data/raw/api/dummyjson_products_raw.json


## 7. Inspect API Metadata

In [12]:
products = api_response.get("products", [])
api_total = api_response.get("total")
api_skip = api_response.get("skip")
api_limit = api_response.get("limit")

print("Products returned:", len(products))
print("API total:", api_total)
print("API skip:", api_skip)
print("API limit:", api_limit)

Products returned: 30
API total: 194
API skip: 0
API limit: 30


## 8. Convert JSON to DataFrame

In [13]:
df_staging = flatten_api_products(api_response)

records_read = len(df_staging)

print("JSON converted to DataFrame.")
print("Records read:", records_read)
print("Column count:", len(df_staging.columns))
print("Columns:", df_staging.columns.tolist())

df_staging.head(5)

JSON converted to DataFrame.
Records read: 30
Column count: 27
Columns: ['id', 'title', 'description', 'category', 'price', 'discountpercentage', 'rating', 'stock', 'tags', 'brand', 'sku', 'weight', 'warrantyinformation', 'shippinginformation', 'availabilitystatus', 'reviews', 'returnpolicy', 'minimumorderquantity', 'images', 'thumbnail', 'dimensions_width', 'dimensions_height', 'dimensions_depth', 'meta_createdat', 'meta_updatedat', 'meta_barcode', 'meta_qrcode']


,id,title,description,category,price,discountpercentage,rating,stock,tags,brand,...,minimumorderquantity,images,thumbnail,dimensions_width,dimensions_height,dimensions_depth,meta_createdat,meta_updatedat,meta_barcode,meta_qrcode
0,1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular...,beauty,9.99,10.48,2.56,99,"[""beauty"", ""mascara""]",Essence,...,48,"[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,15.14,13.08,22.99,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,5784719087687,https://cdn.dummyjson.com/public/qr-code.png
1,2,Eyeshadow Palette with Mirror,The Eyeshadow Palette with Mirror offers a ver...,beauty,19.99,18.19,2.86,34,"[""beauty"", ""eyeshadow""]",Glamour Beauty,...,20,"[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,9.26,22.47,27.67,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,9170275171413,https://cdn.dummyjson.com/public/qr-code.png
2,3,Powder Canister,The Powder Canister is a finely milled setting...,beauty,14.99,9.84,4.64,89,"[""beauty"", ""face powder""]",Velvet Touch,...,22,"[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,29.27,27.93,20.59,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,8418883906837,https://cdn.dummyjson.com/public/qr-code.png
3,4,Red Lipstick,The Red Lipstick is a classic and bold choice ...,beauty,12.99,12.16,4.36,91,"[""beauty"", ""lipstick""]",Chic Cosmetics,...,40,"[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,18.11,28.38,22.17,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,9467746727219,https://cdn.dummyjson.com/public/qr-code.png
4,5,Red Nail Polish,The Red Nail Polish offers a rich and glossy r...,beauty,8.99,11.44,4.32,79,"[""beauty"", ""nail polish""]",Nail Couture,...,22,"[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,21.63,16.48,29.84,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,4063010628104,https://cdn.dummyjson.com/public/qr-code.png


## 9. Define Required and Optional Fields

In [14]:
required_fields = [
    "id",
    "title",
    "description",
    "category",
    "price",
    "rating",
    "stock",
    "sku",
    "availabilitystatus",
]

optional_fields = [
    "brand",
    "tags",
    "images",
    "thumbnail",
    "warrantyinformation",
    "shippinginformation",
    "returnpolicy",
    "minimumorderquantity",
    "dimensions_width",
    "dimensions_height",
    "dimensions_depth",
    "meta_createdat",
    "meta_updatedat",
    "meta_barcode",
    "meta_qrcode",
    "reviews",
]

print("Required fields:", required_fields)
print("Optional fields:", optional_fields)

Required fields: ['id', 'title', 'description', 'category', 'price', 'rating', 'stock', 'sku', 'availabilitystatus']
Optional fields: ['brand', 'tags', 'images', 'thumbnail', 'warrantyinformation', 'shippinginformation', 'returnpolicy', 'minimumorderquantity', 'dimensions_width', 'dimensions_height', 'dimensions_depth', 'meta_createdat', 'meta_updatedat', 'meta_barcode', 'meta_qrcode', 'reviews']


## 10. Validate Required Columns

In [15]:
missing_required_columns = [
    col for col in required_fields
    if col not in df_staging.columns
]

if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

print("Required column validation passed.")

Required column validation passed.


## 11. Check Missing Values

In [16]:
missing_values_all = df_staging.isna().sum()
required_missing_values = df_staging[required_fields].isna().sum()

existing_optional_fields = [
    col for col in optional_fields
    if col in df_staging.columns
]

optional_missing_values = df_staging[existing_optional_fields].isna().sum()

total_missing_values = int(missing_values_all.sum())

print("All missing values:")
print(missing_values_all)

print("\nRequired missing values:")
print(required_missing_values)

print("\nOptional missing values:")
print(optional_missing_values)

print("\nTotal missing values:", total_missing_values)

All missing values:
id                       0
title                    0
description              0
category                 0
price                    0
discountpercentage       0
rating                   0
stock                    0
tags                     0
brand                   15
sku                      0
weight                   0
warrantyinformation      0
shippinginformation      0
availabilitystatus       0
reviews                  0
returnpolicy             0
minimumorderquantity     0
images                   0
thumbnail                0
dimensions_width         0
dimensions_height        0
dimensions_depth         0
meta_createdat           0
meta_updatedat           0
meta_barcode             0
meta_qrcode              0
dtype: int64

Required missing values:
id                    0
title                 0
description           0
category              0
price                 0
rating                0
stock                 0
sku                   0
availabilitystatus  

## 12. Save Parsed Output to Staging

In [17]:
df_staging.to_csv(
    staging_output_path,
    index=False,
    encoding="utf-8"
)

print("Staging saved:", to_relative_path(staging_output_path, PROJECT_ROOT))

Staging saved: data/staging/api/dummyjson_products_staging.csv


## 13. Check Duplicate Rows

In [18]:
duplicate_count = int(df_staging.duplicated().sum())

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


## 14. Create Clean Data

In [19]:
df_clean = df_staging.copy()

df_clean = df_clean.dropna(subset=required_fields)
df_clean = df_clean.drop_duplicates()

records_valid = len(df_clean)
records_invalid = records_read - records_valid

print("Records read:", records_read)
print("Records valid:", records_valid)
print("Records invalid:", records_invalid)

df_clean.head(5)

Records read: 30
Records valid: 30
Records invalid: 0


,id,title,description,category,price,discountpercentage,rating,stock,tags,brand,...,minimumorderquantity,images,thumbnail,dimensions_width,dimensions_height,dimensions_depth,meta_createdat,meta_updatedat,meta_barcode,meta_qrcode
0,1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular...,beauty,9.99,10.48,2.56,99,"[""beauty"", ""mascara""]",Essence,...,48,"[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,15.14,13.08,22.99,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,5784719087687,https://cdn.dummyjson.com/public/qr-code.png
1,2,Eyeshadow Palette with Mirror,The Eyeshadow Palette with Mirror offers a ver...,beauty,19.99,18.19,2.86,34,"[""beauty"", ""eyeshadow""]",Glamour Beauty,...,20,"[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,9.26,22.47,27.67,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,9170275171413,https://cdn.dummyjson.com/public/qr-code.png
2,3,Powder Canister,The Powder Canister is a finely milled setting...,beauty,14.99,9.84,4.64,89,"[""beauty"", ""face powder""]",Velvet Touch,...,22,"[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,29.27,27.93,20.59,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,8418883906837,https://cdn.dummyjson.com/public/qr-code.png
3,4,Red Lipstick,The Red Lipstick is a classic and bold choice ...,beauty,12.99,12.16,4.36,91,"[""beauty"", ""lipstick""]",Chic Cosmetics,...,40,"[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,18.11,28.38,22.17,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,9467746727219,https://cdn.dummyjson.com/public/qr-code.png
4,5,Red Nail Polish,The Red Nail Polish offers a rich and glossy r...,beauty,8.99,11.44,4.32,79,"[""beauty"", ""nail polish""]",Nail Couture,...,22,"[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,21.63,16.48,29.84,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,4063010628104,https://cdn.dummyjson.com/public/qr-code.png


## 15. Save Cleaned Output

In [20]:
df_clean.to_csv(
    clean_output_path,
    index=False,
    encoding="utf-8"
)

print("Clean saved:", to_relative_path(clean_output_path, PROJECT_ROOT))

Clean saved: data/clean/api/dummyjson_products_clean.csv


## 16. Generate Ingestion Log

In [21]:
end_time = datetime.now(timezone.utc).isoformat()

ingestion_log = {
    "run_id": run_id,
    "source_name": source_name,
    "source_type": source_type,
    "input_path_or_url": api_url,
    "start_time": start_time,
    "end_time": end_time,
    "status": status,
    "records_read": int(records_read),
    "records_valid": int(records_valid),
    "records_invalid": int(records_invalid),
    "duplicate_rows_removed": int(duplicate_count),
    "api_total": api_total,
    "api_skip": api_skip,
    "api_limit": api_limit,
    "required_fields": required_fields,
    "optional_fields": optional_fields,
    "missing_values": missing_values_all.astype(int).to_dict(),
    "required_missing_values": required_missing_values.astype(int).to_dict(),
    "optional_missing_values": optional_missing_values.astype(int).to_dict(),
    "total_missing_values": int(total_missing_values),
    "error_message": error_message,
    "raw_output_path": to_relative_path(raw_output_path, PROJECT_ROOT),
    "staging_output_path": to_relative_path(staging_output_path, PROJECT_ROOT),
    "clean_output_path": to_relative_path(clean_output_path, PROJECT_ROOT),
    "owner": owner
}

with open(log_output_path, "w", encoding="utf-8") as file:
    json.dump(ingestion_log, file, indent=4, ensure_ascii=False)

print("Log saved:", to_relative_path(log_output_path, PROJECT_ROOT))
ingestion_log

Log saved: logs/api_ingestion_log.json


{'run_id': '0ef8141c-6f3c-4a3c-b8ed-4931d2dcba6f',
 'source_name': 'dummyjson_products_api',
 'source_type': 'api',
 'input_path_or_url': 'https://dummyjson.com/products',
 'start_time': '2026-07-01T06:32:21.546195+00:00',
 'end_time': '2026-07-01T06:32:22.775754+00:00',
 'status': 'success',
 'records_read': 30,
 'records_valid': 30,
 'records_invalid': 0,
 'duplicate_rows_removed': 0,
 'api_total': 194,
 'api_skip': 0,
 'api_limit': 30,
 'required_fields': ['id',
  'title',
  'description',
  'category',
  'price',
  'rating',
  'stock',
  'sku',
  'availabilitystatus'],
 'optional_fields': ['brand',
  'tags',
  'images',
  'thumbnail',
  'warrantyinformation',
  'shippinginformation',
  'returnpolicy',
  'minimumorderquantity',
  'dimensions_width',
  'dimensions_height',
  'dimensions_depth',
  'meta_createdat',
  'meta_updatedat',
  'meta_barcode',
  'meta_qrcode',
  'reviews'],
 'missing_values': {'id': 0,
  'title': 0,
  'description': 0,
  'category': 0,
  'price': 0,
  'discou

## 17. Final Output Check

In [22]:
print("Raw exists:", raw_output_path.exists())
print("Staging exists:", staging_output_path.exists())
print("Clean exists:", clean_output_path.exists())
print("Log exists:", log_output_path.exists())

print("\nOutput files:")
print(to_relative_path(raw_output_path, PROJECT_ROOT))
print(to_relative_path(staging_output_path, PROJECT_ROOT))
print(to_relative_path(clean_output_path, PROJECT_ROOT))
print(to_relative_path(log_output_path, PROJECT_ROOT))

Raw exists: True
Staging exists: True
Clean exists: True
Log exists: True

Output files:
data/raw/api/dummyjson_products_raw.json
data/staging/api/dummyjson_products_staging.csv
data/clean/api/dummyjson_products_clean.csv
logs/api_ingestion_log.json


## 18. Summary

```text
data/raw/api/dummyjson_products_raw.json
data/staging/api/dummyjson_products_staging.csv
data/clean/api/dummyjson_products_clean.csv
logs/api_ingestion_log.json
```